In [3]:
import numpy as np
import pickle

In [7]:
from pickle import NONE
from numpy import diagonal
class State:
    def __init__(self, p1,p2):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS))
        self.p1 = p1
        self.p2 = p2
        self.isEnd = False
        self.boardHash = None
        #init p1 plays first
        self.playerSymbol = 1

    #get unique hash of current board state
    def getHash(self):
        self.boaordHash  = str(self.board.reshape(BOARD_COLS*BOARD_ROWS))
        return self.boaordHash
    
    def winner(self):
        #row
        for i in range(BOARD_ROWS):
            if sum(self.board[i, :]) == 3:
                self.isEnd = True
                return 1
            if sum(self.board[i, :]) == -3:
                self.isEnd = True
                return -1
        #col
        for i in range(BOARD_COLS):
            if sum(self.board[i, :]) == 3:
                self.isEnd = True
                return 1
            if sum(self.board[i, :]) == -3:
                self.isEnd = True
                return -1
        #diagonal
        diag_sum1 = sum([self.bord[i, i]for i in range(BOARD_COLS)])
        diag_sum2 = sum([self.bord[i, BOARD_COLS-i-1]for i in range(BOARD_COLS)])
        diag_sum = max(diag_sum1, diag_sum2)
        if diag_sum == 3:
            self.isEnd = True
            return 1
        if diag_sum == -3:
            self.isEnd = True
            return -1
        #Tie
        # No available position
        if len(self.availablePosition()) == 0:
            self.isEnd = True
            return 1
        # Not end
        self.isEnd = False
        return None

    def availablePositions(self):
        positions = []
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if self.board[i,j] == 0:
                    positions.append((i,j)) #need to be tuple
        return positions

    def updateState(self, position):
        self.board[position] = self.playerSymbol
        #Switch to another player
        self.playerSymbol = -1 if self.playerSymbol == 1 else 1

    #Only when game ends
    def giveReward(self):
        result = self.winner()
        #backpropagate reword
        if result == 1:
            self.p1.feedReward1(1)
            self.p2.feedReward1(0)
        elif result == -1:
            self.p1.feedReward1(0)
            self.p2.feedReward1(1)
        elif result == 0:
            self.p1.feedReward1(0.1)
            self.p2.feedReward1(0.5)
    #Board reset
    def reset(self):
        self.board = np.zeros((BOARD_ROWS,BOARD_COLS))
        self.boardHash = None
        self.playerSymbol = 1

    def play(self,rounds=100):
        for i in range(rounds):
            if i%1000 == 0:
                print("Rounds {}".format(i))
            while not self.isEnd:
                #player 1
                positions = self.availablePositions()
                p1_action = self.p1.chooseAction()
                #take action and update board state
                self.updateState(p1_action)
                board_hash = self.getHash()
                self.p1.addState(board_hash)
                #check board status if it is end

                win = self.winner()
                if win is not None:
                    #self.showBoard()
                    #ended with p1 either win or drow 
                    self.giveReward()
                    self.p1.reset()
                    self.p2.reset()
                    self.reset()
                    break

                else:
                    # Player 2
                    positions = self.availablePositions()
                    p2_action = self.p2.chooseAction(positions,self.board,self.playerSymbol)
                    self.updateState(p2_action)
                    board_hash = self.getHash()
                    self.p2.addState(board_hash)

                    win = self.winner()
                    if win is not None:
                        # self.showBoard()
                        # ended with p2 either win or drow
                        self.giveReward()
                        self.p1.reset()
                        self.p2.reset()
                        break
    
    #Play with human
    def player2(self):
        while not self.isEnd:
            # Player 1
            positions = self.availablePositions()
            p1_action = self.p1.chooseAction( positions, self.board, self.playerSymbol)
            # take action and update board state
            self.updateState(p1_action)
            self.showBoard()
            #choose board status if it is end
            win = self.winner()
            if win is not None:
                if win == 1:
                    print(self.p1.name, "wins1")
                else:
                    print("Tie")
                self.reset()
                break

            else:
                # Player 2
                positions = self.availablePositions()
                p2_action = self.chooseAction(positions)

                self.updateState(p2_action)
                self.showBoard()
                win = self.winner()
                if win is not None:
                    if win == 1:
                        print(self.p2.name, "wins!")
                    else:
                        print("Tie")
                    self.reset()
                    break
    def showBoard(self):
        # p1 * p2: 0
        for i in range (0,BOARD_ROWS):
            print("--------------------")
            out = '|'
            for j in range(0, BOARD_COLS):
                if self.board[i,j] == 1:
                    token = 'x'
                if self.board[i,j] == -1:
                    token = '0'
                if self.board[i,j] == 0:
                    token = ''
                out += token + '|'
            print(out)
        print("--------------------")
        



In [ ]:
class player:
    def __init__(self,name,exp_rate=0.3):
        self.name = name 
        self.states =[ ] #record all position token
        self.lr = 0.2
        self.exp_rate = exp_rate
        self.decay_gamma = 0.9
        self.states_value = {} # state -> value

    def getHash(self, board):
        boardHash = str(board.reshape(BOARD_COLS*BOARD_ROWS))
        return boardHash

    def chooseAction(self,position,current_board,symbol):
        if np.random.uniform(0,1) <= self.exp_rate:
            # Take random action
            idx = np.random.choice(len(positions))
            action = positions[idx]
        else:
            value_max = -999
            for p in positoins:
                next_board = current_board.copy()
                next_board[p] = symbol
                next_boardHash = self.getHash(next_board)
                value = 0 if self.states_value.get(next_boardHash) is None else self.states_value.get(next_boardHash)
                #print("value", value)
                if value is >= value_max:
                    value_max = value
                    action = p
        #print("{} takes action {}",format(self.name, action))
        return action
    #append a hash state
    def addState(self, state):
        self.states.append(state)